# **Purpose**

**Task:** context + answer → question

This notebook fine-tunes `gpt2` (a decoder-only / causal language model) on a subset of SQuAD to generate a question given a context passage and an answer span. Swap `MODEL_NAME` to `gpt2-medium` or `gpt2-large` to compare model scales.

**Runtime:** Runtime → Change runtime type → GPU (T4 is suitable for `gpt2`).

## **Install dependencies**

In [1]:
!pip install -qU \
 transformers==5.16.1 \
 accelerate==1.14.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.7 MB/s eta 0:00:00


## **Imports**

In [2]:
import numpy as np
import torch
import transformers
import accelerate
import os

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)

# Optional: Kaggle Secrets (Uncomment if executing on Kaggle)
from kaggle_secrets import UserSecretsClient

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

torch: 2.10.0+cu128
transformers: 5.16.1
accelerate: 1.14.0


## **Load the API Keys and Tokens**

In [3]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")   # must match the exact secret name we set in Kaggle Secrets

os.environ["HF_TOKEN"] = hf_token

## **Config**

Tweak these for your experiment. TRAIN_SUBSET_SIZE / VAL_SUBSET_SIZE control how much of SQuAD you use - start small to sanity-check the pipeline before scaling up.

`MAX_INPUT_LENGTH` bounds the prompt (context + answer) tokens and `MAX_TARGET_LENGTH` bounds the question tokens; unlike the seq2seq BART setup, GPT-2 sees prompt and question concatenated in a single sequence, so `MAX_INPUT_LENGTH + MAX_TARGET_LENGTH` is the effective max sequence length.

In [4]:
MODEL_NAME = "openai-community/gpt2"              # try "gpt2-medium" or "gpt2-large" if compute allows
MAX_INPUT_LENGTH = 512           # context + answer prompt length
MAX_TARGET_LENGTH = 96           # generated question length
TRAIN_SUBSET_SIZE = 10000        # set to None for full SQuAD train split
VAL_SUBSET_SIZE = 10
TRAIN_EPOCH_SIZE = 4
OUTPUT_DIR = "/content/gpt2-qg"
SEED = 42
# *******Change the Huggingface pushing path below**********

## **Load SQuAD and take a subset**

Uses the Hugging Face squad dataset. We can swap to "squad_v2" if we also want unanswerable examples (note: squad_v2 has empty answer lists for some examples, which the preprocessing below already handles gracefully).

In [5]:
raw = load_dataset("squad")

train_ds = raw["train"].shuffle(seed=SEED)
val_ds = raw["validation"].shuffle(seed=SEED)

if TRAIN_SUBSET_SIZE:
    train_ds = train_ds.select(range(TRAIN_SUBSET_SIZE))
if VAL_SUBSET_SIZE:
    val_ds = val_ds.select(range(VAL_SUBSET_SIZE))

print(train_ds)
print(val_ds)

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 10000
})
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 10
})


In [6]:
# Peek at one example
train_ds[0]

{'id': '573173d8497a881900248f0c',
 'title': 'Egypt',
 'context': 'The Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan independent agency of the US government, has placed Egypt on its watch list of countries that require close monitoring due to the nature and extent of violations of religious freedom engaged in or tolerated by the government. According to a 2010 Pew Global Attitudes survey, 84% of Egyptians polled supported the death penalty for those who leave Islam; 77% supported whippings and cutting off of hands for theft and robbery; and 82% support stoning a person who commits adultery.',
 'question': 'What percentage of Egyptians polled support death penalty for those leaving Islam?',
 'answers': {'text': ['84%'], 'answer_start': [468]}}

## **Load tokenizer and model**

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# GPT-2 has no pad token by default - reuse eos as pad for batching
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## **Preprocessing**

Builds the same prompt as the BART notebook, then appends a `Question:` marker followed by the gold question and an end-of-sequence token. Since GPT-2 is a causal LM trained on a single concatenated sequence, we mask out the prompt tokens in `labels` (set to `-100`) so the loss is only computed on the question tokens - this keeps the model focused on learning to generate questions rather than to reproduce the context.

In [8]:
def preprocess(examples):
    input_ids_list = []
    labels_list = []
    attention_mask_list = []

    for context, answers, question in zip(examples["context"], examples["answers"], examples["question"]):
        answer_text = answers["text"][0] if len(answers["text"]) > 0 else ""
        prompt = (
            f"Target Answer: {answer_text}\n"
            f"Generate a question from the following context where the target answer is the correct answer. "
            f"Do not include phrases like \'According to the text\' in the question and do not repeat the context in the question.\n"
            f"Context: {context}\n"
            f"Question:"
        )
        completion = f" {question}{tokenizer.eos_token}"

        prompt_ids = tokenizer(prompt, truncation=True, max_length=MAX_INPUT_LENGTH)["input_ids"]
        completion_ids = tokenizer(completion, truncation=True, max_length=MAX_TARGET_LENGTH)["input_ids"]

        input_ids = prompt_ids + completion_ids
        labels = [-100] * len(prompt_ids) + completion_ids

        input_ids_list.append(input_ids)
        labels_list.append(labels)
        attention_mask_list.append([1] * len(input_ids))

    return {
        "input_ids": input_ids_list,
        "labels": labels_list,
        "attention_mask": attention_mask_list,
    }

tokenized_train = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)


class CausalLMQGCollator:
    """Pads variable-length (input_ids, attention_mask, labels) batches for causal-LM fine-tuning.

    Unlike DataCollatorForSeq2Seq, this pads labels with -100 (not tokenizer.pad_token_id) so
    padded and prompt positions are both ignored by the loss.
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        batch_input_ids, batch_attention, batch_labels = [], [], []

        for f in features:
            pad_len = max_len - len(f["input_ids"])
            batch_input_ids.append(f["input_ids"] + [self.tokenizer.pad_token_id] * pad_len)
            batch_attention.append(f["attention_mask"] + [0] * pad_len)
            batch_labels.append(f["labels"] + [-100] * pad_len)

        return {
            "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(batch_attention, dtype=torch.long),
            "labels": torch.tensor(batch_labels, dtype=torch.long),
        }


data_collator = CausalLMQGCollator(tokenizer=tokenizer)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

## **Training arguments**

On Colab, set fp16=True if you have a T4/V100/A100 GPU (NVIDIA mixed precision). Adjust batch size / gradient accumulation if you hit out-of-memory errors. GPT-2 typically tolerates a slightly higher learning rate than BART.

In [9]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    save_strategy="epoch",
    learning_rate=5e-5,                # bump to 1e-4 if convergence is slow
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    num_train_epochs=TRAIN_EPOCH_SIZE,
    fp16=True,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    processing_class=tokenizer,
    data_collator=data_collator,
)

## **Train the model**

In [10]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,2.385866
100,2.224530
150,2.152611
200,2.108322
250,2.047701
300,2.064100
350,1.856345
400,1.773508
450,1.754836
500,1.799038


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1252, training_loss=1.7443346960095172, metrics={'train_runtime': 2675.6245, 'train_samples_per_second': 14.95, 'train_steps_per_second': 0.468, 'total_flos': 7542970214400000.0, 'train_loss': 1.7443346960095172, 'epoch': 4.0})

## **Save the fine-tuned model**

In [11]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/gpt2-qg


## **Push the model to Huggingface Hub**

In [12]:
model.push_to_hub("gaurav-dey/gpt2-qg")
tokenizer.push_to_hub("gaurav-dey/gpt2-qg")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/gaurav-dey/gpt2-qg/commit/49242563f80100ba6d6b67e8902bb17390016fef', commit_message='Upload tokenizer', commit_description='', oid='49242563f80100ba6d6b67e8902bb17390016fef', pr_url=None, repo_url=RepoUrl('https://huggingface.co/gaurav-dey/gpt2-qg', endpoint='https://huggingface.co', repo_type='model', repo_id='gaurav-dey/gpt2-qg'), pr_revision=None, pr_num=None)

## **Sanity-check generations**

For a causal LM, generation continues from the prompt rather than decoding from a separate encoder output, so we only decode the newly generated tokens (everything after the prompt) to recover the generated question.

In [13]:
sample = val_ds.select(range(5))
for ex in sample:
    context = ex["context"]
    gold_question = ex["question"]
    answer_text = ex["answers"]["text"][0] if ex["answers"]["text"] else ""

    prompt = (
        f"Target Answer: {answer_text}\n"
        f"Generate a question from the following context where the target answer is the correct answer. "
        f"Do not include phrases like \'According to the text\' in the question and do not repeat the context in the question.\n"
        f"Context: {context}\n"
        f"Question:"
    )

    input_ids = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH
    ).input_ids.to(model.device)
    prompt_len = input_ids.shape[1]

    output_ids = model.generate(
        input_ids,
        max_new_tokens=MAX_TARGET_LENGTH,
        num_beams=4,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_question = tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()

    print(f"Answer:              {answer_text}")
    print(f"Gold question:       {gold_question}")
    print(f"Generated question:  {generated_question}")
    print("-" * 80)

Answer:              1852
Gold question:       In what year did Massachusetts first require children to be educated in schools?
Generated question:  In what year did the Massachusetts legislature set standards for private schooling?
--------------------------------------------------------------------------------
Answer:              1962
Gold question:       When were stromules discovered?
Generated question:  In what year was chloroplast membranes first observed?
--------------------------------------------------------------------------------
Answer:              Horace Walpole
Gold question:       Which artist who had a major influence on the Gothic Revival is represented in the V&A's British galleries?
Generated question:  Who influenced the Gothic Revival?
--------------------------------------------------------------------------------
Answer:              several regional colleges and universities
Gold question:       In 1890, who did the university decide to team up with?
Generat